# ML Model Lifecycle for Production Pricing

**A Professional Guide to Machine Learning in Quantitative Finance**

This tutorial demonstrates the complete lifecycle of an ML pricing model in a production environment:

1. **Architecture Overview** - Understanding the package structure
2. **Configuration-Driven Training** - Reproducible experiment setup
3. **Data Engineering** - Building and validating datasets
4. **Model Selection** - Comparing architectures
5. **Training Pipeline** - Professional training with callbacks
6. **Evaluation & Benchmarking** - Validating against analytic models
7. **Hyperparameter Tuning** - Finding optimal configurations
8. **Model Deployment** - Saving, loading, and inference
9. **Production Monitoring** - Tracking performance over time
10. **Advanced Topics** - Portfolio models and calibration

---

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import time
from datetime import datetime

# TensorFlow
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

# QuantStrata ML imports
from src.machine_learning.core.config import (
    TrainingConfig, OptimizerConfig, EarlyStoppingConfig,
    LRScheduleConfig, ModelConfig, CheckpointConfig
)
from src.machine_learning.core.types import (
    TrainingResult, EvaluationResult, TuningResult
)
from src.machine_learning.data.dataset import (
    TFDataset, NormalizationStats, create_pricing_dataset
)
from src.machine_learning.data.pricing.build import (
    build_pricing_data, PricingDataResult
)
from src.machine_learning.models.pricing.model import (
    MLPPricer, ResidualMLPPricer, create_mlp_pricer
)
from src.machine_learning.training.trainer import (
    Trainer, TrainingResult as TrainerResult, fit_model
)
from src.machine_learning.pipelines.evaluation import (
    evaluate_model, METRIC_FUNCTIONS
)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

---
## Section 0: Architecture Overview

### Package Structure

The `machine_learning` package follows a modular architecture that separates:
- **Generic infrastructure** (pipelines, training, evaluation)
- **Model-specific components** (data builders, model architectures)

```
src/machine_learning/
├── core/                    # Base classes and protocols
│   ├── base.py             # BaseModel, PricingModel, CalibrationModel
│   ├── config.py           # TrainingConfig, OptimizerConfig, etc.
│   ├── protocols.py        # Trainable protocol
│   └── types.py            # Result dataclasses
├── data/                    # Data utilities
│   ├── dataset.py          # TFDataset, NormalizationStats
│   └── pricing/            # Model-specific data builders
│       └── build.py        # build_pricing_data()
├── models/                  # Model architectures
│   └── pricing/            # Model-specific implementations
│       └── model.py        # MLPPricer, ResidualMLPPricer
├── training/               # Training infrastructure
│   └── trainer.py          # Trainer class
├── pipelines/              # Generic pipelines
│   ├── training.py         # run_training()
│   └── evaluation.py       # evaluate_model()
└── inference/              # Deployment utilities
    ├── model_io.py         # save_model, load_model
    └── predictor.py        # Predictor class
```

In [ ]:
# Visualize the ML Pipeline
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    ML MODEL LIFECYCLE PIPELINE                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║   ┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────┐  ║
║   │    DATA     │───▶│    MODEL    │───▶│  TRAINING   │───▶│ EVALUATION  │  ║
║   │  BUILDER    │    │  CREATION   │    │   LOOP      │    │  METRICS    │  ║
║   └─────────────┘    └─────────────┘    └─────────────┘    └─────────────┘  ║
║         │                  │                  │                  │          ║
║         ▼                  ▼                  ▼                  ▼          ║
║   tf.data.Dataset    MLPPricer         TrainingResult    EvaluationResult   ║
║   NormStats          PricingModel      Checkpoints       Greeks Accuracy    ║
║                                                                              ║
║   ┌─────────────┐    ┌─────────────┐    ┌─────────────┐    ┌─────────────┐  ║
║   │   TUNING    │───▶│ DEPLOYMENT  │───▶│  INFERENCE  │───▶│ MONITORING  │  ║
║   │   SEARCH    │    │  ARTIFACTS  │    │   BATCH     │    │   DRIFT     │  ║
║   └─────────────┘    └─────────────┘    └─────────────┘    └─────────────┘  ║
║         │                  │                  │                  │          ║
║         ▼                  ▼                  ▼                  ▼          ║
║   TuningResult       SavedModel          Predictor        Retrain Trigger   ║
║   Best Config        Metadata            Latency              Alerts        ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

---
## Section 1: Configuration-Driven Training

Professional ML systems use **configuration files** for reproducibility and experiment tracking.

### Key Principles:
- Separate **model config** (architecture) from **training config** (hyperparameters)
- All configs are serializable to JSON/YAML
- Track configs alongside results for reproducibility

In [ ]:
# Define experiment configuration as a dictionary (easily serializable to YAML)
EXPERIMENT_CONFIG = {
    "experiment_name": "pricing_mlp_v1",
    "description": "MLP pricer for European vanilla options",
    
    "data": {
        "n_samples": 50000,
        "train_ratio": 0.7,
        "val_ratio": 0.15,
        "test_ratio": 0.15,
        "normalize": True,
        "seed": 42,
    },
    
    "model": {
        "type": "MLPPricer",
        "hidden_units": [128, 64, 32],
        "activation": "relu",
        "dropout_rate": 0.1,
        "use_batch_norm": True,
        "kernel_regularizer": 0.0001,
    },
    
    "training": {
        "epochs": 100,
        "batch_size": 256,
        "optimizer": {
            "name": "adam",
            "learning_rate": 0.001,
        },
        "lr_schedule": {
            "type": "cosine",
            "warmup_steps": 500,
        },
        "early_stopping": {
            "patience": 15,
            "min_delta": 1e-5,
            "restore_best_weights": True,
        },
        "loss": "mse",
        "metrics": ["mae"],
    },
}

print("Experiment Configuration:")
print(json.dumps(EXPERIMENT_CONFIG, indent=2))

In [ ]:
# Create configuration objects from the dictionary
def create_training_config(cfg: dict) -> TrainingConfig:
    """Create TrainingConfig from experiment config dict."""
    train_cfg = cfg["training"]
    
    return TrainingConfig(
        epochs=train_cfg["epochs"],
        batch_size=train_cfg["batch_size"],
        optimizer=OptimizerConfig(
            name=train_cfg["optimizer"]["name"],
            learning_rate=train_cfg["optimizer"]["learning_rate"],
        ),
        early_stopping=EarlyStoppingConfig(
            patience=train_cfg["early_stopping"]["patience"],
            min_delta=train_cfg["early_stopping"]["min_delta"],
            restore_best_weights=train_cfg["early_stopping"]["restore_best_weights"],
        ),
        loss=train_cfg["loss"],
        metrics=train_cfg["metrics"],
        seed=cfg["data"]["seed"],
        verbose=1,
    )

training_config = create_training_config(EXPERIMENT_CONFIG)
print(f"Training Config: {training_config.epochs} epochs, batch_size={training_config.batch_size}")
print(f"Optimizer: {training_config.optimizer.name}, LR={training_config.optimizer.learning_rate}")

---
## Section 2: Data Engineering for Pricing Models

### Data Pipeline Contract:
- Data builders output `tf.data.Dataset` for efficient training
- Normalization statistics are saved for inference
- Train/val/test splits are reproducible with seed

In [ ]:
# Build the pricing dataset
data_cfg = EXPERIMENT_CONFIG["data"]

data_result = build_pricing_data(
    n_samples=data_cfg["n_samples"],
    train_ratio=data_cfg["train_ratio"],
    val_ratio=data_cfg["val_ratio"],
    test_ratio=data_cfg["test_ratio"],
    batch_size=training_config.batch_size,
    seed=data_cfg["seed"],
    normalize=data_cfg["normalize"],
)

print(f"Dataset built successfully!")
print(f"Metadata: {data_result.metadata}")
print(f"Feature normalization: mean shape = {data_result.feature_stats.mean.shape}")

In [ ]:
# Inspect a sample batch
for features, targets in data_result.train_ds.take(1):
    print(f"Feature batch shape: {features.shape}")
    print(f"Target batch shape: {targets.shape}")
    print(f"\nFirst sample features (normalized):")
    feature_names = ["spot", "strike", "vol", "rate", "expiry", "is_call"]
    for name, val in zip(feature_names, features[0].numpy()):
        print(f"  {name}: {val:.4f}")
    print(f"  Target (price): {targets[0].numpy()[0]:.4f}")

In [ ]:
# Data Quality Checks
def validate_dataset(data_result: PricingDataResult):
    """Run data quality checks."""
    checks = []
    
    # Check for NaN/Inf
    for features, targets in data_result.train_ds.take(10):
        has_nan = tf.reduce_any(tf.math.is_nan(features)) or tf.reduce_any(tf.math.is_nan(targets))
        has_inf = tf.reduce_any(tf.math.is_inf(features)) or tf.reduce_any(tf.math.is_inf(targets))
        if has_nan.numpy() or has_inf.numpy():
            checks.append(("NaN/Inf Check", "FAILED"))
            break
    else:
        checks.append(("NaN/Inf Check", "PASSED"))
    
    # Check normalization
    if data_result.feature_stats is not None:
        checks.append(("Feature Normalization", "ENABLED"))
    else:
        checks.append(("Feature Normalization", "DISABLED"))
    
    # Check dataset sizes
    train_samples = sum(1 for _ in data_result.train_ds.unbatch())
    val_samples = sum(1 for _ in data_result.val_ds.unbatch())
    test_samples = sum(1 for _ in data_result.test_ds.unbatch())
    checks.append(("Train samples", train_samples))
    checks.append(("Validation samples", val_samples))
    checks.append(("Test samples", test_samples))
    
    print("\nData Quality Report:")
    print("=" * 40)
    for check, result in checks:
        print(f"  {check}: {result}")
    return checks

validate_dataset(data_result)

---
## Section 3: Model Selection & Architecture

### Available Architectures:
- **MLPPricer**: Standard feedforward network with optional batch norm and dropout
- **ResidualMLPPricer**: Deep residual network for complex pricing surfaces

### Key Features:
- Automatic Greeks computation via `tf.GradientTape`
- Keras-serializable for deployment
- Configurable via dataclasses

In [ ]:
# Create models for comparison
model_cfg = EXPERIMENT_CONFIG["model"]

# Standard MLP
mlp_model = MLPPricer(
    hidden_units=model_cfg["hidden_units"],
    activation=model_cfg["activation"],
    dropout_rate=model_cfg["dropout_rate"],
    use_batch_norm=model_cfg["use_batch_norm"],
    kernel_regularizer=model_cfg["kernel_regularizer"],
    name="mlp_pricer_v1",
)

# Residual MLP
residual_model = ResidualMLPPricer(
    n_blocks=3,
    block_units=64,
    dropout_rate=0.1,
    name="residual_pricer_v1",
)

# Build models with sample input
sample_input = tf.random.uniform((1, 6))
_ = mlp_model(sample_input)
_ = residual_model(sample_input)

print("MLP Model Summary:")
print(mlp_model.summary_dict())
print(f"\nResidual Model Summary:")
print(residual_model.summary_dict())

In [ ]:
# Demonstrate Greeks computation
# Create a sample option: ATM call with 1Y expiry
test_option = tf.constant([
    [100.0, 100.0, 0.2, 0.05, 1.0, 1.0],  # ATM call
    [100.0, 110.0, 0.2, 0.05, 1.0, 1.0],  # OTM call
    [100.0, 90.0, 0.2, 0.05, 1.0, 0.0],   # ITM put
], dtype=tf.float32)

# Normalize using saved stats
test_option_norm = (test_option - data_result.feature_stats.mean) / (data_result.feature_stats.std + 1e-8)

# Get price and Greeks
greeks = mlp_model.price_with_greeks(test_option_norm)

print("Greeks for test options (before training - random weights):")
print(f"{'Option':<15} {'Price':>10} {'Delta':>10} {'Gamma':>10} {'Vega':>10}")
print("-" * 60)
option_names = ["ATM Call", "OTM Call", "ITM Put"]
for i, name in enumerate(option_names):
    print(f"{name:<15} {greeks['price'][i,0].numpy():>10.4f} {greeks['delta'][i,0].numpy():>10.4f} "
          f"{greeks['gamma'][i,0].numpy():>10.4f} {greeks['vega'][i,0].numpy():>10.4f}")

---
## Section 4: Training Pipeline

### Professional Training Features:
- **Early stopping** with best weight restoration
- **Learning rate scheduling** (warmup + cosine decay)
- **Model checkpointing** for fault tolerance
- **TensorBoard logging** for visualization
- **Mixed precision** for faster training on GPUs

In [ ]:
# Create the Trainer
trainer = Trainer(mlp_model, training_config)

# Compile with custom metrics
trainer.compile(
    loss='mse',
    metrics=['mae'],
)

print(f"Model compiled with:")
print(f"  Loss: {training_config.loss}")
print(f"  Metrics: {training_config.metrics}")
print(f"  Optimizer: {training_config.optimizer.name}")

In [ ]:
# Train the model
print("Starting training...")
start_time = time.time()

result = trainer.fit(
    train_data=data_result.train_ds,
    val_data=data_result.val_ds,
)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.1f}s")
print(f"Final epoch: {result.final_epoch}")
print(f"Best epoch: {result.best_epoch}")
print(f"Best validation loss: {result.best_val_loss:.6f}")
print(f"Stopped early: {result.stopped_early}")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1 = axes[0]
epochs = range(1, len(result.history['loss']) + 1)
ax1.plot(epochs, result.history['loss'], 'b-', label='Training Loss', linewidth=2)
if 'val_loss' in result.history:
    ax1.plot(epochs, result.history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
ax1.axvline(result.best_epoch, color='green', linestyle='--', label=f'Best Epoch ({result.best_epoch})', alpha=0.7)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss (MSE)', fontsize=12)
ax1.set_title('Training Progress', fontsize=14)
ax1.legend()
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# MAE plot
ax2 = axes[1]
if 'mae' in result.history:
    ax2.plot(epochs, result.history['mae'], 'b-', label='Training MAE', linewidth=2)
if 'val_mae' in result.history:
    ax2.plot(epochs, result.history['val_mae'], 'r-', label='Validation MAE', linewidth=2)
ax2.axvline(result.best_epoch, color='green', linestyle='--', label=f'Best Epoch ({result.best_epoch})', alpha=0.7)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('MAE', fontsize=12)
ax2.set_title('Mean Absolute Error', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Section 5: Evaluation & Benchmarking

### Evaluation Metrics:
- **MSE/MAE/RMSE**: Standard regression metrics
- **R²**: Coefficient of determination
- **MAPE**: Mean absolute percentage error
- **Pricing Error**: Comparison vs analytic benchmark (Black-Scholes)

In [ ]:
# Evaluate on test set
from src.machine_learning.core.protocols import KerasTrainableAdapter

# Wrap model for evaluation pipeline
model_adapter = KerasTrainableAdapter(mlp_model)

# Extract test data
test_features = []
test_targets = []
for x, y in data_result.test_ds.unbatch():
    test_features.append(x.numpy())
    test_targets.append(y.numpy())
test_features = np.stack(test_features)
test_targets = np.stack(test_targets)

# Evaluate
eval_result = evaluate_model(
    model_adapter,
    test_features,
    test_targets,
    metrics=["mse", "mae", "rmse", "r2", "mape"],
    training_result=None,
    metadata={"experiment": EXPERIMENT_CONFIG["experiment_name"]},
)

print(eval_result.summary())

In [ ]:
# Residual Analysis
predictions = mlp_model.predict(test_features, verbose=0)
residuals = test_targets - predictions

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Predicted vs Actual
ax1 = axes[0]
ax1.scatter(test_targets, predictions, alpha=0.3, s=10)
min_val, max_val = test_targets.min(), test_targets.max()
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
ax1.set_xlabel('Actual Price (normalized)', fontsize=12)
ax1.set_ylabel('Predicted Price (normalized)', fontsize=12)
ax1.set_title('Predicted vs Actual', fontsize=14)
ax1.legend()

# Residual Distribution
ax2 = axes[1]
ax2.hist(residuals.flatten(), bins=50, edgecolor='black', alpha=0.7)
ax2.axvline(0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Residual', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title(f'Residual Distribution (mean={residuals.mean():.4f})', fontsize=14)

# Residual vs Predicted
ax3 = axes[2]
ax3.scatter(predictions, residuals, alpha=0.3, s=10)
ax3.axhline(0, color='red', linestyle='--', linewidth=2)
ax3.set_xlabel('Predicted Price', fontsize=12)
ax3.set_ylabel('Residual', fontsize=12)
ax3.set_title('Residual vs Predicted', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Greeks accuracy validation (post-training)
greeks_trained = mlp_model.price_with_greeks(test_option_norm)

print("Greeks for test options (AFTER training):")
print(f"{'Option':<15} {'Price':>10} {'Delta':>10} {'Gamma':>10} {'Vega':>10}")
print("-" * 60)
for i, name in enumerate(option_names):
    print(f"{name:<15} {greeks_trained['price'][i,0].numpy():>10.4f} "
          f"{greeks_trained['delta'][i,0].numpy():>10.4f} "
          f"{greeks_trained['gamma'][i,0].numpy():>10.4f} "
          f"{greeks_trained['vega'][i,0].numpy():>10.4f}")

---
## Section 6: Hyperparameter Tuning

Systematic search over architecture and training hyperparameters.

In [ ]:
# Define hyperparameter search space
TUNING_SPACE = {
    "hidden_units": [
        [64, 32],
        [128, 64],
        [128, 64, 32],
    ],
    "learning_rate": [0.01, 0.001, 0.0001],
    "dropout_rate": [0.0, 0.1, 0.2],
}

def run_tuning_trial(hidden_units, learning_rate, dropout_rate, epochs=20):
    """Run a single tuning trial."""
    # Create model
    model = MLPPricer(
        hidden_units=hidden_units,
        dropout_rate=dropout_rate,
        use_batch_norm=True,
    )
    
    # Create config
    config = TrainingConfig(
        epochs=epochs,
        batch_size=256,
        optimizer=OptimizerConfig(name="adam", learning_rate=learning_rate),
        early_stopping=EarlyStoppingConfig(patience=5),
        verbose=0,
    )
    
    # Train
    trainer = Trainer(model, config)
    result = trainer.fit(data_result.train_ds, data_result.val_ds)
    
    return {
        "config": {
            "hidden_units": hidden_units,
            "learning_rate": learning_rate,
            "dropout_rate": dropout_rate,
        },
        "score": result.best_val_loss,
        "best_epoch": result.best_epoch,
    }

print(f"Tuning space: {len(TUNING_SPACE['hidden_units']) * len(TUNING_SPACE['learning_rate']) * len(TUNING_SPACE['dropout_rate'])} configurations")

In [ ]:
# Run a subset of tuning trials (for demo - full tuning would take longer)
import itertools

# Sample a few configurations for demo
demo_configs = [
    ([64, 32], 0.001, 0.0),
    ([128, 64], 0.001, 0.1),
    ([128, 64, 32], 0.0001, 0.1),
]

trials = []
print("Running tuning trials...\n")

for i, (hu, lr, dr) in enumerate(demo_configs):
    print(f"Trial {i+1}/{len(demo_configs)}: hidden_units={hu}, lr={lr}, dropout={dr}")
    trial_result = run_tuning_trial(hu, lr, dr, epochs=15)
    trials.append(trial_result)
    print(f"  -> Val loss: {trial_result['score']:.6f}, Best epoch: {trial_result['best_epoch']}\n")

# Find best configuration
best_trial = min(trials, key=lambda x: x["score"])
print("\n" + "=" * 50)
print("BEST CONFIGURATION:")
print(f"  Hidden units: {best_trial['config']['hidden_units']}")
print(f"  Learning rate: {best_trial['config']['learning_rate']}")
print(f"  Dropout rate: {best_trial['config']['dropout_rate']}")
print(f"  Validation loss: {best_trial['score']:.6f}")

In [ ]:
# Create TuningResult
tuning_result = TuningResult(
    best_config=best_trial["config"],
    best_score=best_trial["score"],
    trials=trials,
    metadata={
        "search_strategy": "grid_sample",
        "n_trials": len(trials),
        "timestamp": datetime.now().isoformat(),
    },
)

print("Tuning Result saved:")
print(json.dumps(tuning_result.to_dict(), indent=2))

---
## Section 7: Model Deployment

### Deployment Artifacts:
- **SavedModel**: Full model with weights and config
- **Metadata**: Experiment config, training results, normalization stats
- **Version tracking**: Model versioning for rollback

In [ ]:
# Create deployment directory
from pathlib import Path
import shutil

DEPLOY_DIR = Path("./model_artifacts")
MODEL_VERSION = "v1.0.0"
MODEL_DIR = DEPLOY_DIR / f"pricing_mlp_{MODEL_VERSION}"

# Clean and create
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
MODEL_DIR.mkdir(parents=True)

print(f"Deployment directory: {MODEL_DIR}")

In [ ]:
# Save model
model_path = MODEL_DIR / "model.keras"
mlp_model.save(str(model_path))
print(f"Model saved to: {model_path}")

# Save normalization stats
stats_path = MODEL_DIR / "normalization_stats.json"
with open(stats_path, "w") as f:
    json.dump({
        "feature_stats": data_result.feature_stats.to_dict(),
        "target_stats": data_result.target_stats.to_dict() if data_result.target_stats else None,
    }, f, indent=2)
print(f"Normalization stats saved to: {stats_path}")

# Save experiment config
config_path = MODEL_DIR / "experiment_config.json"
with open(config_path, "w") as f:
    json.dump(EXPERIMENT_CONFIG, f, indent=2)
print(f"Experiment config saved to: {config_path}")

# Save training result
result_path = MODEL_DIR / "training_result.json"
result.to_json(str(result_path))
print(f"Training result saved to: {result_path}")

# Save metadata
metadata = {
    "model_name": "pricing_mlp",
    "version": MODEL_VERSION,
    "created_at": datetime.now().isoformat(),
    "framework": f"tensorflow-{tf.__version__}",
    "evaluation_metrics": eval_result.metrics,
    "feature_names": ["spot", "strike", "volatility", "rate", "time_to_expiry", "is_call"],
}
metadata_path = MODEL_DIR / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to: {metadata_path}")

# List all artifacts
print("\nDeployment artifacts:")
for f in sorted(MODEL_DIR.iterdir()):
    print(f"  {f.name}")

In [ ]:
# Load model for inference
loaded_model = tf.keras.models.load_model(str(model_path))

# Load normalization stats
with open(stats_path, "r") as f:
    loaded_stats = json.load(f)
loaded_feature_stats = NormalizationStats.from_dict(loaded_stats["feature_stats"])

print("Model loaded successfully!")
print(f"Model name: {loaded_model.name}")

In [ ]:
# Create a simple Predictor class
class OptionPricer:
    """Production-ready option pricer."""
    
    def __init__(self, model_dir: Path):
        self.model_dir = Path(model_dir)
        
        # Load model
        self.model = tf.keras.models.load_model(str(self.model_dir / "model.keras"))
        
        # Load normalization stats
        with open(self.model_dir / "normalization_stats.json") as f:
            stats = json.load(f)
        self.feature_stats = NormalizationStats.from_dict(stats["feature_stats"])
        self.target_stats = NormalizationStats.from_dict(stats["target_stats"]) if stats.get("target_stats") else None
        
        # Load metadata
        with open(self.model_dir / "metadata.json") as f:
            self.metadata = json.load(f)
    
    def price(self, spot, strike, vol, rate, expiry, is_call):
        """Price a single option."""
        features = np.array([[spot, strike, vol, rate, expiry, float(is_call)]], dtype=np.float32)
        return self.price_batch(features)[0]
    
    def price_batch(self, features: np.ndarray) -> np.ndarray:
        """Price a batch of options."""
        # Normalize
        features_norm = self.feature_stats.normalize(features)
        
        # Predict
        predictions_norm = self.model.predict(features_norm, verbose=0)
        
        # Denormalize
        if self.target_stats:
            predictions = self.target_stats.denormalize(predictions_norm)
        else:
            predictions = predictions_norm
        
        return predictions.flatten()
    
    def info(self):
        """Return model information."""
        return self.metadata

# Use the pricer
pricer = OptionPricer(MODEL_DIR)
print("Pricer loaded:")
print(json.dumps(pricer.info(), indent=2))

In [ ]:
# Test inference
test_prices = [
    pricer.price(spot=100, strike=100, vol=0.2, rate=0.05, expiry=1.0, is_call=True),
    pricer.price(spot=100, strike=110, vol=0.2, rate=0.05, expiry=1.0, is_call=True),
    pricer.price(spot=100, strike=90, vol=0.2, rate=0.05, expiry=1.0, is_call=False),
]

print("\nSample Predictions:")
print(f"ATM Call (S=100, K=100): ${test_prices[0]:.4f}")
print(f"OTM Call (S=100, K=110): ${test_prices[1]:.4f}")
print(f"ITM Put (S=100, K=90):   ${test_prices[2]:.4f}")

---
## Section 8: Production Monitoring

### Key Monitoring Metrics:
- **Inference latency**: Time per prediction
- **Prediction drift**: Changes in output distribution
- **Feature drift**: Changes in input distribution
- **Error rate**: Comparison with benchmark models

In [ ]:
# Latency benchmark
import time

def benchmark_latency(pricer, n_samples=1000, batch_sizes=[1, 10, 100, 1000]):
    """Benchmark inference latency."""
    results = []
    
    for batch_size in batch_sizes:
        # Generate random inputs
        features = np.random.uniform(
            low=[80, 80, 0.1, 0.01, 0.1, 0],
            high=[120, 120, 0.5, 0.10, 2.0, 1],
            size=(batch_size, 6)
        ).astype(np.float32)
        
        # Warmup
        _ = pricer.price_batch(features)
        
        # Benchmark
        n_runs = max(1, n_samples // batch_size)
        start = time.perf_counter()
        for _ in range(n_runs):
            _ = pricer.price_batch(features)
        elapsed = time.perf_counter() - start
        
        total_predictions = n_runs * batch_size
        latency_per_pred = (elapsed / total_predictions) * 1000  # ms
        throughput = total_predictions / elapsed
        
        results.append({
            "batch_size": batch_size,
            "latency_ms": latency_per_pred,
            "throughput_per_sec": throughput,
        })
    
    return results

latency_results = benchmark_latency(pricer)
print("Inference Latency Benchmark:")
print(f"{'Batch Size':<12} {'Latency (ms)':<15} {'Throughput (preds/s)':<20}")
print("-" * 50)
for r in latency_results:
    print(f"{r['batch_size']:<12} {r['latency_ms']:<15.4f} {r['throughput_per_sec']:<20.0f}")

In [ ]:
# Prediction drift monitoring
class DriftMonitor:
    """Monitor prediction drift over time."""
    
    def __init__(self, baseline_mean: float, baseline_std: float, threshold: float = 2.0):
        self.baseline_mean = baseline_mean
        self.baseline_std = baseline_std
        self.threshold = threshold  # Number of std deviations
        self.history = []
    
    def check(self, predictions: np.ndarray) -> dict:
        """Check for drift in predictions."""
        current_mean = predictions.mean()
        current_std = predictions.std()
        
        z_score = (current_mean - self.baseline_mean) / (self.baseline_std + 1e-8)
        drift_detected = abs(z_score) > self.threshold
        
        result = {
            "timestamp": datetime.now().isoformat(),
            "current_mean": current_mean,
            "current_std": current_std,
            "z_score": z_score,
            "drift_detected": drift_detected,
        }
        self.history.append(result)
        
        return result

# Initialize monitor with baseline statistics
baseline_preds = predictions.flatten()
monitor = DriftMonitor(
    baseline_mean=baseline_preds.mean(),
    baseline_std=baseline_preds.std(),
    threshold=2.0,
)

# Simulate monitoring over time
print("Drift Monitoring Simulation:")
print(f"Baseline: mean={baseline_preds.mean():.4f}, std={baseline_preds.std():.4f}")
print("\nChecking batches...")

for i in range(5):
    # Simulate new predictions (with slight drift in later batches)
    drift_factor = 1.0 + i * 0.1  # Gradual drift
    simulated_preds = baseline_preds * drift_factor + np.random.normal(0, 0.01, len(baseline_preds))
    
    result = monitor.check(simulated_preds)
    status = "ALERT" if result["drift_detected"] else "OK"
    print(f"  Batch {i+1}: z_score={result['z_score']:.2f}, status={status}")

---
## Section 9: Advanced Topics

### Extension Points:
- **Portfolio-level models**: GNN-RNN hybrid for correlated trades
- **Calibration models**: Learning model parameters from market data
- **Risk integration**: Connecting with the risk pipeline

In [ ]:
# Example: Extending to a CalibrationModel
from src.machine_learning.core.base import CalibrationModel

class HestonCalibrator(CalibrationModel):
    """
    Neural network for Heston model calibration.
    
    Input: Implied volatility surface (flattened)
    Output: Heston parameters [v0, kappa, theta, sigma, rho]
    """
    
    def __init__(self, n_iv_points: int = 50, **kwargs):
        super().__init__(
            name="heston_calibrator",
            target_model="heston",
            n_parameters=5,
            **kwargs,
        )
        self.n_iv_points = n_iv_points
        
        # Architecture
        self.encoder = tf.keras.Sequential([
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dense(32, activation='relu'),
        ])
        self.param_head = tf.keras.layers.Dense(5)  # 5 Heston params
    
    @property
    def parameter_names(self):
        return ["v0", "kappa", "theta", "sigma", "rho"]
    
    def call(self, inputs, training=False):
        x = self.encoder(inputs, training=training)
        return self.param_head(x)

# Create and test
calibrator = HestonCalibrator(n_iv_points=50)
sample_iv_surface = tf.random.uniform((1, 50))  # Flattened IV surface
params = calibrator(sample_iv_surface)

print("Heston Calibrator Output:")
for name, val in zip(calibrator.parameter_names, params[0].numpy()):
    print(f"  {name}: {val:.4f}")

In [ ]:
# Integration with Risk Pipeline (conceptual)
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                    ML + RISK PIPELINE INTEGRATION                            ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║   Market Data    ┌─────────────┐    ┌─────────────┐    ┌─────────────┐      ║
║   ─────────────▶│  ML Pricer  │───▶│  Risk       │───▶│  VaR /      │      ║
║                  │  (fast)     │    │  Scenarios  │    │  Stress     │      ║
║   Portfolio      └─────────────┘    └─────────────┘    └─────────────┘      ║
║   ─────────────▶                                                             ║
║                                                                              ║
║   Use Cases:                                                                 ║
║   • Real-time portfolio valuation with ML models                             ║
║   • Fast Greeks for risk attribution                                         ║
║   • Scenario generation with ML-based sensitivities                          ║
║   • VaR computation at scale                                                 ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

---
## Summary

This tutorial covered the complete ML model lifecycle for production pricing:

| Stage | Key Components | Output |
|-------|---------------|--------|
| **Data** | `build_pricing_data()` | `tf.data.Dataset`, `NormalizationStats` |
| **Model** | `MLPPricer`, `ResidualMLPPricer` | Keras model with Greeks |
| **Training** | `Trainer`, `TrainingConfig` | `TrainingResult`, checkpoints |
| **Evaluation** | `evaluate_model()` | `EvaluationResult`, metrics |
| **Tuning** | Grid/random search | `TuningResult`, best config |
| **Deployment** | SavedModel, metadata | Production artifacts |
| **Monitoring** | Drift detection, latency | Alerts, metrics |

### Next Steps:
1. Extend to exotic options (barriers, digitals)
2. Implement GNN-RNN hybrid for portfolio pricing
3. Add model calibration pipeline
4. Integrate with backtesting framework
5. Deploy to production with MLOps

In [ ]:
# Cleanup
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)
    print(f"Cleaned up {MODEL_DIR}")

print("\nTutorial complete!")